# Spike C — Vocos (mel → waveform) → ONNX

**Goal:** export `charactr/vocos-mel-24khz` to ONNX and verify PyTorch vs
ORT CPU parity at audio-sample RMS < 0.05 on 3 test mels.

Vocos is the vocoder that turns 100-dim mel spectrograms into 24 kHz
waveforms. Both F5 and ZipVoice produce mels; Vocos is shared between
them and published at `/shared/vocos.onnx` in the deployed architecture.

This is the easiest of the three export spikes — Vocos is a simple
conv/transformer stack with no rotary embeddings and no flow-matching
conditioning. If this fails, something is seriously wrong.

## Runtime

- Colab free T4 or CPU-only works; Vocos is tiny (~60M params).
- ~20 min wall clock.

## Artifacts

- `vocos.onnx` (~60 MB FP16) — shared across voices
- `parity_report_vocos.json`

In [ ]:
!pip install -q torch==2.4.0 torchaudio==2.4.0
!pip install -q onnx==1.16.2 onnxruntime==1.19.2 onnxconverter-common==1.14.0
!pip install -q vocos==0.1.0

import os, sys, json, hashlib, datetime
import torch
import numpy as np
from vocos import Vocos

ART_DIR = '/content/spike_c_artifacts'
os.makedirs(ART_DIR, exist_ok=True)
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 1. Load + trace Vocos

In [ ]:
vocos = Vocos.from_pretrained('charactr/vocos-mel-24khz').eval()
print('Params:', sum(p.numel() for p in vocos.parameters()) / 1e6, 'M')

# Vocos expects mel spectrograms [B, n_mels=100, T]. Dummy input for export.
# 200 frames ≈ 2 seconds of audio at hop=256, sr=24000.
B, N_MEL, T = 1, 100, 200
dummy_mel = torch.randn(B, N_MEL, T)

In [ ]:
# Wrap Vocos so torch.onnx.export sees a clean forward().
class VocosWrapper(torch.nn.Module):
    def __init__(self, vocos):
        super().__init__()
        self.vocos = vocos
    def forward(self, mel):
        # Returns waveform [B, T_audio]
        return self.vocos.decode(mel)

model = VocosWrapper(vocos).eval()

# Sanity: PyTorch forward works.
with torch.no_grad():
    pt_wav = model(dummy_mel)
print('PyTorch output shape:', tuple(pt_wav.shape))

## 2. Export to ONNX

In [ ]:
onnx_path = f'{ART_DIR}/vocos.onnx'

torch.onnx.export(
    model,
    dummy_mel,
    onnx_path,
    input_names=['mel'],
    output_names=['audio'],
    dynamic_axes={
        'mel': {2: 'n_frames'},
        'audio': {1: 'n_samples'},
    },
    opset_version=17,
    do_constant_folding=True,
)
size_mb = os.path.getsize(onnx_path) / 1024 / 1024
print(f'Exported → {onnx_path}  ({size_mb:.1f} MB)')

In [ ]:
# Convert FP32 → FP16. Vocos is small so this is optional, but it matches the
# FP16 precision we'll use for the generator ONNX in Spike A.
from onnxconverter_common import float16
import onnx

m = onnx.load(onnx_path)
m_fp16 = float16.convert_float_to_float16(m, keep_io_types=True)
fp16_path = f'{ART_DIR}/vocos_fp16.onnx'
onnx.save(m_fp16, fp16_path)
print(f'FP16 size: {os.path.getsize(fp16_path) / 1024 / 1024:.1f} MB')

## 3. Parity check — PyTorch vs ORT CPU

Three test mels. For the vocoder we compare waveforms directly (not mels)
since the whole point is mel → audio. RMS < 0.05 on the normalized
waveform is our bar.

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
print('ORT inputs:', [i.name for i in sess.get_inputs()])
print('ORT outputs:', [o.name for o in sess.get_outputs()])

torch.manual_seed(0)
test_mels = [
    torch.randn(1, 100, 150),   # short
    torch.randn(1, 100, 300),   # medium
    torch.randn(1, 100, 500),   # long
]

parity = []
for i, mel in enumerate(test_mels):
    with torch.no_grad():
        pt_wav = model(mel).numpy()
    ort_wav = sess.run(None, {'mel': mel.numpy().astype('float32')})[0]

    # Length may differ by a few samples due to conv padding edge effects;
    # compare on the shared prefix.
    n = min(pt_wav.shape[-1], ort_wav.shape[-1])
    rms = float(np.sqrt(np.mean((pt_wav[..., :n] - ort_wav[..., :n]) ** 2)))
    passed = rms < 0.05
    parity.append({'test': i, 'frames': mel.shape[-1], 'rms': rms, 'pass': passed})
    print(f'  mel #{i} (frames={mel.shape[-1]}): RMS={rms:.5f} → {"PASS" if passed else "FAIL"}')

## 4. Write parity report

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(2**20), b''):
            h.update(chunk)
    return h.hexdigest()

report = {
    'spike': 'C',
    'generated_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'rms_threshold': 0.05,
    'artifacts': {
        'vocos_fp32': {
            'path': onnx_path,
            'size_mb': os.path.getsize(onnx_path) / 1024 / 1024,
            'sha256': sha256(onnx_path),
        },
        'vocos_fp16': {
            'path': fp16_path,
            'size_mb': os.path.getsize(fp16_path) / 1024 / 1024,
            'sha256': sha256(fp16_path),
        },
    },
    'parity': parity,
    'all_passed': all(p['pass'] for p in parity),
}

report_path = f'{ART_DIR}/parity_report_vocos.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print('Wrote', report_path)
print(json.dumps(report, indent=2, default=str))

In [ ]:
try:
    from google.colab import files
    files.download(report_path)
    files.download(fp16_path)
except ImportError:
    print('Download manually from', ART_DIR)

## 5. What to record in REPORT.md

Spike C section of `spikes/REPORT.md`:
- FP32 and FP16 sizes
- RMS parity per test mel
- SHA-256 of final `vocos_fp16.onnx` for reproducibility
- Any ops that required opset bumps (don't expect any; Vocos is vanilla)

The `vocos_fp16.onnx` artifact also gets placed next to Spike B's
`index.html` so the harness can demonstrate end-to-end mel → audio.